# Services, Boot, systemd & Packages

## The boot sequence — power-on to login prompt

Between pressing the power button and seeing the login prompt, four well-defined stages happen. Knowing them helps when something fails to boot.

```
   ┌─────────────────────────────────────────┐
1. │  BIOS / UEFI firmware (hardware ROM)    │   POST, find boot device
   └─────────────────────────────────────────┘
                  ↓
   ┌─────────────────────────────────────────┐
2. │  Bootloader (GRUB)                      │   load kernel + initramfs
   └─────────────────────────────────────────┘
                  ↓
   ┌─────────────────────────────────────────┐
3. │  Linux kernel + initramfs               │   probe hardware, mount real /
   └─────────────────────────────────────────┘
                  ↓
   ┌─────────────────────────────────────────┐
4. │  systemd (PID 1)                        │   start services → login prompt
   └─────────────────────────────────────────┘
```

### Stage 1 — BIOS / UEFI

When you power on, the **firmware** runs first. The firmware lives on a chip on the motherboard; you don't choose it, the hardware does.

- **BIOS** (Basic Input/Output System) — the legacy firmware. Loads the first 512 bytes of the boot disk (the **MBR**) and jumps to that code.
- **UEFI** (Unified Extensible Firmware Interface) — the modern replacement. Reads a small **EFI System Partition** (ESP, usually mounted at `/boot/efi`) containing `.efi` bootloader files, and runs the one specified in its boot menu.

Almost all hardware made since ~2012 supports UEFI. Many still allow legacy BIOS mode for backwards compatibility. **You can usually tell which is in use**:

```bash
[ -d /sys/firmware/efi ] && echo "UEFI" || echo "BIOS"
```

The firmware also runs the **POST** (Power-On Self-Test) — basic hardware checks — before handing off to the bootloader.

### Stage 2 — GRUB

**GRUB** ("GRand Unified Bootloader") is the bootloader on virtually all Linux systems. Its job: present a menu (if more than one OS is installed), load the chosen kernel into memory, and start it.

The config lives in `/boot/grub/grub.cfg` — but **don't edit that file directly**. It's regenerated from `/etc/default/grub` plus scripts in `/etc/grub.d/`:

```bash
sudo vim /etc/default/grub                  # main config knobs
sudo update-grub                            # Debian/Ubuntu: regenerate grub.cfg
sudo grub2-mkconfig -o /boot/grub2/grub.cfg # RHEL/Fedora: same idea
```

Common edits:

- **`GRUB_TIMEOUT=5`** — how long to wait at the menu before booting the default.
- **`GRUB_DEFAULT=0`** — which menu entry to boot by default.
- **`GRUB_CMDLINE_LINUX="quiet splash"`** — kernel command-line arguments. Adding `nomodeset` here is a common GPU-driver fix; adding `single` boots into single-user mode for recovery.

At the GRUB menu, you can press `e` to edit a boot entry temporarily — useful when a config change breaks normal boot. Append `single` to the linux line, Ctrl-X to boot.

### Stage 3 — kernel + initramfs

GRUB loads two files into memory:

- The **kernel** (`/boot/vmlinuz-<version>`) — Linux itself.
- The **initramfs** (`/boot/initrd.img-<version>` or `/boot/initramfs-*.img`) — a small in-memory filesystem containing just enough to find and mount the real root filesystem.

Why initramfs exists: your real root filesystem might be on an LVM volume on an encrypted disk on a RAID array. The kernel needs drivers + tools to assemble those layers before it can mount `/`. The initramfs ships those drivers and tools. Once `/` is mounted, the initramfs is discarded.

To rebuild the initramfs (e.g. after adding kernel modules):

```bash
sudo update-initramfs -u                    # Debian/Ubuntu
sudo dracut -f                              # RHEL/Fedora
```

### Stage 4 — systemd

Once the kernel has mounted the real `/`, it executes **`/sbin/init`** as PID 1. On every modern distro, that's a symlink to **`systemd`**. From here, systemd takes over — starting services, mounting filesystems, configuring the network, and ultimately reaching the **default target** (usually `graphical.target` or `multi-user.target`), which means "everything's ready, here's a login prompt."

The rest of this notebook is mostly about that fourth stage.

## What systemd is, in one minute

**systemd** is the **init system** — PID 1 — on every major modern Linux distribution. It's also a suite of related daemons (`journald` for logs, `logind` for login sessions, `networkd` for networking, `timesyncd` for NTP, `resolved` for DNS).

What systemd does:

- Starts services in dependency order, in parallel where possible.
- Manages service lifecycles — restarts crashed services, runs them on schedules, activates them on demand.
- Tracks processes via **cgroups** (kernel control groups) — so killing a service kills all its children too.
- Aggregates logs into the **journal**.
- Handles boot targets ("what should be running for a graphical desktop session?").

Everything systemd manages is a **unit**. Units are described by text files in two main locations:

| Location | What's there |
|---|---|
| **`/lib/systemd/system/`** or **`/usr/lib/systemd/system/`** | **Vendor-supplied** units. Packages drop their service files here. Don't edit. |
| **`/etc/systemd/system/`** | **Local/admin overrides** and locally-written units. Higher priority than `/lib`. **Edit here.** |
| **`~/.config/systemd/user/`** | Per-user units (managed with `systemctl --user`). |

You'll mostly read existing units in `/lib/` and write new ones (or drop-in overrides) in `/etc/`.

## Unit types

Every unit file ends in a suffix indicating its **type**:

| Suffix | What it is | Example |
|---|---|---|
| **`.service`** | a daemon or one-shot process to run | `nginx.service` |
| **`.socket`** | a socket the kernel listens on; activates the matching `.service` on demand | `ssh.socket` |
| **`.timer`** | a timer that triggers a unit (the modern cron replacement) | `apt-daily.timer` |
| **`.target`** | a "group" — used to organise other units, like runlevels | `multi-user.target` |
| **`.mount`** | a mount point (alternative to /etc/fstab) | `home.mount` |
| **`.automount`** | mount on first access | `media-usb.automount` |
| **`.device`** | a hardware device (auto-generated from udev) | `dev-sda1.device` |
| **`.path`** | trigger a unit when a file/directory changes | `mywatcher.path` |
| **`.slice`** | resource control group (cgroup) | `user.slice` |
| **`.swap`** | a swap file or partition | `dev-sdb2.swap` |

The most common ones you'll write and manage: **`.service`**, **`.timer`**, and occasionally **`.target`**.

To see all units of a type:

```bash
systemctl list-units --type=service                # running services
systemctl list-units --type=service --all          # all services, including stopped
systemctl list-units --type=timer                  # active timers
systemctl list-unit-files --type=service           # all service unit files (enabled or not)
```

## `systemctl` — the universal verb

`systemctl` is the one command you'll use for almost every systemd interaction. Its shape is:

```
systemctl VERB [UNIT...]
```

The verbs you'll use day-to-day:

| Verb | What it does |
|---|---|
| **`status`** | show current state of a unit, plus recent log lines |
| **`start`** | start a unit now (temporary — won't auto-start at next boot unless enabled) |
| **`stop`** | stop a unit now |
| **`restart`** | stop then start |
| **`reload`** | tell the service to re-read its config (if it supports it) — does NOT restart |
| **`reload-or-restart`** | try reload, fall back to restart |
| **`enable`** | mark unit to start at boot |
| **`disable`** | unmark — won't start at boot |
| **`enable --now`** | enable AND start now (saves a command) |
| **`disable --now`** | disable AND stop now |
| **`is-active`** | quick scriptable check: prints "active" or "inactive" |
| **`is-enabled`** | scriptable: prints "enabled" / "disabled" / "static" |
| **`is-failed`** | scriptable: prints "failed" or "active" |
| **`cat`** | show the unit file contents (vendor + drop-ins) |
| **`edit`** | edit a drop-in for the unit (creates `/etc/systemd/system/UNIT.d/override.conf`) |
| **`edit --full`** | edit the full unit file (copies vendor unit to `/etc/`) |
| **`daemon-reload`** | re-read all unit files from disk — **required after creating/editing units** |
| **`list-units`** | show currently loaded units |
| **`list-unit-files`** | show all known unit files |
| **`list-dependencies`** | show what a unit depends on |
| **`mask`** | completely block a unit — even other units can't start it |
| **`unmask`** | undo `mask` |

### The most important rule

**After creating or editing a unit file, run `systemctl daemon-reload`** before trying to start/restart the service. systemd caches unit definitions; without daemon-reload it won't see your changes.

If you change something and the service doesn't pick it up, your first check is "did I daemon-reload?" Second is "did I `systemctl restart` (not just `reload`)?"

In [ ]:
!systemctl list-units --type=service --state=running 2>/dev/null | head -10
!systemctl is-active sshd 2>/dev/null || systemctl is-active ssh 2>/dev/null

## Reading `systemctl status`

`systemctl status SERVICE` is the first command to run when investigating any service. The output looks like:

```
● nginx.service - A high performance web server and a reverse proxy server
     Loaded: loaded (/lib/systemd/system/nginx.service; enabled; vendor preset: enabled)
     Active: active (running) since Mon 2026-06-03 09:14:22 UTC; 2h 17min ago
       Docs: man:nginx(8)
   Main PID: 1234 (nginx)
      Tasks: 3 (limit: 4684)
     Memory: 12.4M
     CGroup: /system.slice/nginx.service
             ├─1234 nginx: master process /usr/sbin/nginx
             ├─1235 nginx: worker process
             └─1236 nginx: worker process

Jun 03 09:14:22 server systemd[1]: Started A high performance web server...
Jun 03 09:14:22 server nginx[1234]: Configuration loaded OK
Jun 03 11:31:55 server systemd[1]: Reloaded A high performance web server...
```

Decode it line by line:

- **`●`** — green dot means active; red means failed; white means inactive
- **`Loaded:`** — where the unit file is + whether it's `enabled` (starts at boot) or `disabled`
- **`Active:`** — runtime state. **`active (running)`**, `active (exited)` (one-shot completed OK), `inactive (dead)`, `failed`, `activating`, `deactivating`
- **`Main PID:`** — the primary process; useful for `ps -p`, `lsof -p`, etc.
- **`Tasks: 3 (limit: 4684)`** — how many processes/threads under this service; the hard limit from cgroup config
- **`Memory:`** — current RSS attributable to this service (sum of cgroup)
- **`CGroup:`** — the process tree managed by this service. All children, no matter how nested
- **Last 10 log lines** — recent journal entries for this service. Use `journalctl -u nginx` for more.

Key insight: **`systemctl status` shows the latest log lines for you.** When a service won't start, that's where you look first.

### Common states and what to do

| Active state | Meaning | Next step |
|---|---|---|
| `active (running)` | working as expected | nothing |
| `active (exited)` | one-shot finished successfully | nothing — `Type=oneshot` is normal |
| `inactive (dead)` | not running, no errors | `systemctl start` if you want it |
| `failed` | crashed or exited with non-zero | `journalctl -u UNIT` to find why |
| `activating (auto-restart)` | crashed, waiting to restart | check logs; may be a config error in a restart loop |

In [ ]:
!systemctl status systemd-journald 2>/dev/null | head -15

## Writing a basic service unit

Time to write one. A `.service` file is INI-style (sections in `[brackets]`, `key=value` lines). A minimal but complete service that runs a script:

```ini
# /etc/systemd/system/myapp.service
[Unit]
Description=My Application
After=network-online.target
Wants=network-online.target

[Service]
Type=simple
ExecStart=/usr/local/bin/myapp
Restart=on-failure
RestartSec=5s
User=myapp
Group=myapp
WorkingDirectory=/var/lib/myapp
Environment="APP_ENV=production"

[Install]
WantedBy=multi-user.target
```

The three sections:

**`[Unit]`** — generic info and dependencies.

- **`Description=`** — human description (shown in `systemctl status`).
- **`After=`** / **`Before=`** — ordering. **`After=network-online.target`** = "don't start me until the network is up."
- **`Wants=`** / **`Requires=`** — dependencies. `Wants` is soft (start if you can); `Requires` is hard (fail me if my dependency fails).
- **`Documentation=`** — `man:foo(8)` or `https://...` — shown in status.

**`[Service]`** — how to run the program.

- **`Type=`** — the most important field. Common values:
  - **`simple`** — `ExecStart` is the main process, runs in foreground. The default.
  - **`forking`** — `ExecStart` forks and exits; the daemon is a child. For programs that "daemonise" themselves.
  - **`oneshot`** — runs and exits, no long-running process. Pair with `RemainAfterExit=yes` for "setup" units.
  - **`notify`** — process sends a "ready" signal to systemd when it's truly ready.
  - **`dbus`** — service registers on D-Bus when ready.
- **`ExecStart=`** — the command to run. Must be an **absolute path**.
- **`ExecStop=`**, **`ExecReload=`** — optional commands for stop/reload.
- **`Restart=`** — `no` / `on-failure` / `always` / `on-abnormal`. The killer feature: automatic restart on crash.
- **`RestartSec=`** — wait this long before restarting.
- **`User=`**, **`Group=`** — run as this user/group instead of root.
- **`WorkingDirectory=`** — `cd` here before running.
- **`Environment=`** — set environment variables. Multiple `Environment=` lines are allowed.
- **`EnvironmentFile=`** — load variables from a file (often `/etc/default/myapp`).
- **`StandardOutput=`**, **`StandardError=`** — where stdout/stderr go. Default is `journal` (best — captured by journald).

**`[Install]`** — what happens when you `systemctl enable`.

- **`WantedBy=multi-user.target`** — when enabled, this unit becomes a "wanted by" of `multi-user.target`, meaning it'll start whenever `multi-user.target` is reached (i.e. at boot).

### Activating your new unit

```bash
sudo systemctl daemon-reload                # tell systemd to read the new file
sudo systemctl start myapp                  # try it now
sudo systemctl status myapp                 # is it running?
sudo journalctl -u myapp -f                 # watch its logs live
sudo systemctl enable myapp                 # start at boot
# or both at once:
sudo systemctl enable --now myapp
```

### A timer to schedule it

Pair the service with a **`.timer`** for the cron replacement:

```ini
# /etc/systemd/system/myapp.timer
[Unit]
Description=Run myapp daily at 2am

[Timer]
OnCalendar=*-*-* 02:00:00              # cron-like calendar spec
Persistent=true                        # run after boot if the time was missed

[Install]
WantedBy=timers.target
```

Activate the timer (not the service — the timer triggers the service):

```bash
sudo systemctl enable --now myapp.timer
systemctl list-timers                       # see all timers, when they next fire
```

`OnCalendar=` supports rich syntax: `daily`, `weekly`, `monthly`, `*-*-* 02:00:00`, `Mon..Fri 09:00`. Test a spec with `systemd-analyze calendar 'Mon 14:00'`.

## Drop-ins — modifying a vendor unit without editing it

When a package ships a `.service` file you want to tweak (raise a memory limit, add an environment variable, change restart policy), **don't edit the vendor file**. It'll be overwritten on the next package update.

Instead, create a **drop-in**: a small override file in `/etc/systemd/system/UNIT.d/`.

The easy way is **`systemctl edit`**:

```bash
sudo systemctl edit nginx
```

This opens your editor on `/etc/systemd/system/nginx.service.d/override.conf` — initially empty. Add only the sections you want to change:

```ini
[Service]
Environment="WORKER_PROCESSES=8"
LimitNOFILE=65536
```

Save and exit. The override is merged with the vendor unit (overriding matching keys, adding new ones). systemd automatically runs `daemon-reload`. You still need `systemctl restart nginx` for the new config to take effect.

To see the merged result:

```bash
systemctl cat nginx                         # shows vendor + drop-ins concatenated
sudo systemctl show nginx                   # shows the effective merged values
```

To revert (delete the drop-in):

```bash
sudo systemctl revert nginx
```

For LFCS: **prefer drop-ins over full edits**. They survive package updates, are easy to audit, and easy to remove.

## Targets — systemd's runlevels

Older init systems had **runlevels** — numbered system states (0 = halt, 3 = multi-user text, 5 = graphical). systemd's equivalent is **targets** — units that group other units to define a system state.

The targets you'll meet:

| Target | What it means | Old runlevel |
|---|---|---|
| **`poweroff.target`** | halt and power off | 0 |
| **`rescue.target`** | single-user mode, root shell on console, no network | 1 |
| **`multi-user.target`** | multi-user, network, no GUI — **server default** | 3 |
| **`graphical.target`** | multi-user, network, GUI — **desktop default** | 5 |
| **`reboot.target`** | restart the system | 6 |
| **`emergency.target`** | minimal mode — only `/` mounted, root shell, no services | (none) |
| **`default.target`** | symlink to one of the above; what boots by default | (none) |

To see what's running, switch, or set defaults:

```bash
systemctl get-default                        # what target boots by default
sudo systemctl set-default multi-user.target # change boot default
sudo systemctl isolate rescue.target         # switch to rescue NOW (drops out of GUI)
sudo systemctl isolate graphical.target      # switch back
```

You can also reach `emergency.target` and `rescue.target` from GRUB at boot time by appending `systemd.unit=rescue.target` to the kernel command line — useful when normal boot is broken.

For LFCS, know `multi-user.target`, `graphical.target`, `rescue.target`, `emergency.target`, and `systemctl get-default` / `set-default` / `isolate`.

### Shutdown and reboot

Three equivalent ways to reboot a systemd machine:

```bash
sudo systemctl reboot                       # the modern way
sudo reboot                                 # shorthand
sudo shutdown -r now                        # the traditional way (still works)
```

And to power off:

```bash
sudo systemctl poweroff
sudo poweroff
sudo shutdown -h now
sudo shutdown -h +5 "Updating kernel"       # schedule with a wall-broadcast message
sudo shutdown -c                            # cancel a scheduled shutdown
```

In [ ]:
!systemctl get-default
!systemctl list-units --type=target --state=active 2>/dev/null

## `journald` and `journalctl` — the systemd log system

systemd ships with its own log daemon, **`systemd-journald`**, which collects log messages from everything — the kernel, every service, every user session — into a **binary structured journal**. The tool to read it is **`journalctl`**.

The headline benefits over the traditional `tail /var/log/*.log` approach:

- **One unified log** — everything in one place, with metadata (which service, which user, which PID, severity).
- **Powerful filtering** — by unit, time range, priority, boot, fields. No more grepping eight files.
- **Persistent OR volatile** — default is volatile (lost on reboot); easy to make persistent.

### Basic `journalctl` commands

```bash
journalctl                                  # entire journal, oldest first (pages with less)
journalctl -e                               # jump to the end
journalctl -f                               # follow, live, like tail -f
journalctl -n 50                            # last 50 entries
journalctl -r                               # reverse (newest first)
journalctl --no-pager                       # just dump to stdout
```

### Filtering by service

```bash
journalctl -u nginx                         # all entries for nginx.service
journalctl -u nginx -f                      # follow nginx live
journalctl -u nginx --since "1 hour ago"    # last hour
journalctl -u nginx --since today
journalctl -u nginx --since "2026-06-01" --until "2026-06-02"
```

### Filtering by priority

The journal stores syslog-style priorities (0 = emerg, 7 = debug). Filter with `-p`:

```bash
journalctl -p err                           # errors and above (err, crit, alert, emerg)
journalctl -p warning..err                  # range
journalctl -p err -u nginx                  # errors from nginx
```

### Filtering by boot

```bash
journalctl --list-boots                     # list previous boots with offsets and times
journalctl -b                               # current boot only
journalctl -b -1                            # the previous boot
journalctl -b 0 -p err                      # errors from current boot
```

### Filtering by kernel / by PID / by user / by command

```bash
journalctl -k                               # kernel messages only (replaces dmesg in many ways)
journalctl _PID=1234                        # entries from PID 1234
journalctl _UID=1000                        # entries from UID 1000
journalctl /usr/sbin/sshd                   # entries from a specific binary
```

### Persistent vs volatile

By default on many distros, `journald` writes to `/run/log/journal/` — **tmpfs, lost on reboot**. To make persistent:

```bash
sudo mkdir -p /var/log/journal
sudo systemctl restart systemd-journald
```

Once `/var/log/journal/` exists, `journald` automatically uses it. Or set `Storage=persistent` in `/etc/systemd/journald.conf`.

To control disk usage:

```bash
# In /etc/systemd/journald.conf:
SystemMaxUse=1G              # journal will use at most 1 GB
SystemKeepFree=500M          # always leave this much free on the FS
SystemMaxFileSize=100M       # individual journal files capped at this
MaxRetentionSec=1month       # discard entries older than this
```

To manually trim:

```bash
sudo journalctl --vacuum-time=2weeks        # delete entries older than 2 weeks
sudo journalctl --vacuum-size=500M          # trim down to 500 MB total
```

For LFCS: **know `journalctl -u SERVICE`, `-f`, `--since`, `-p`, `-b`, and the persistent/volatile distinction.**

In [ ]:
!journalctl -n 5 --no-pager 2>/dev/null
!journalctl -u systemd-journald --no-pager 2>/dev/null | tail -5

## Traditional logs — `/var/log/` and `rsyslog`

Even on systemd systems, **traditional plain-text logs in `/var/log/`** are still common. Two reasons:

1. Many services (web servers, databases) write their own logs directly to files in `/var/log/`, not the journal.
2. `rsyslog` (or `syslog-ng`) reads from the journal and writes traditional syslog files alongside, for tools and admins that expect them.

Files you'll see in `/var/log/`:

| File | Contents |
|---|---|
| **`syslog`** (Debian) / **`messages`** (RHEL) | general system messages |
| **`auth.log`** (Debian) / **`secure`** (RHEL) | authentication events (logins, sudo, ssh) |
| **`kern.log`** | kernel messages (also via `dmesg`) |
| **`dpkg.log`** (Debian) / **`dnf.log`** (RHEL) | package install/remove/update history |
| **`boot.log`** | boot messages |
| **`nginx/`** , **`apache2/`** , **`mysql/`** , ... | per-service subdirectories |

Tools to read them:

```bash
sudo tail -f /var/log/syslog                # follow live
sudo less /var/log/auth.log                 # paged view
sudo grep "Failed password" /var/log/auth.log | tail -20
dmesg                                       # kernel ring buffer (no sudo on recent kernels for non-root)
dmesg -w                                    # follow kernel messages
dmesg -T                                    # human-readable timestamps
```

### `logrotate`

Logs that grow forever fill disks. **`logrotate`** rotates them — renames the active log, optionally compresses it, deletes old rotations, restarts the service if needed.

Config: `/etc/logrotate.conf` plus per-service drop-ins in `/etc/logrotate.d/`. A typical drop-in:

```
/var/log/nginx/*.log {
    daily
    rotate 14
    compress
    delaycompress
    missingok
    notifempty
    create 0640 www-data adm
    sharedscripts
    postrotate
        nginx -s reopen
    endscript
}
```

Each option explained:

- **`daily`** — rotate every day (also `weekly`, `monthly`, `size 10M`)
- **`rotate 14`** — keep 14 rotated copies
- **`compress`** — gzip rotated files
- **`delaycompress`** — don't compress the most recent rotation (lets the service finish writing to it)
- **`missingok`** — don't error if the file doesn't exist
- **`notifempty`** — don't rotate if the file is empty
- **`create`** — recreate the file with specific perms/owner after rotation
- **`postrotate`/`endscript`** — commands to run after rotation (here: tell nginx to reopen its log file)

`logrotate` is usually triggered daily by a systemd timer (`logrotate.timer`) or a cron job.

For LFCS, know that `journalctl` is the modern log tool, traditional logs live in `/var/log/`, `dmesg` for kernel ring buffer, and `logrotate` handles rotation.

## Package management — `apt` and `dnf`

Linux software is installed via a **package manager** — a tool that downloads software from a **repository** (a server hosting `.deb` or `.rpm` files plus a catalog), resolves dependencies, and installs cleanly with the ability to remove later.

Two major families:

- **Debian / Ubuntu** — `.deb` packages, **`apt`** is the modern CLI (with `dpkg` as the lower-level tool).
- **RHEL / Fedora / CentOS / Alma / Rocky** — `.rpm` packages, **`dnf`** is the modern CLI (with `rpm` as the lower-level tool).

The vocabulary maps closely between the two. Here's the parity table for everything you'll do:

| Task | apt (Debian/Ubuntu) | dnf (RHEL/Fedora) |
|---|---|---|
| Refresh package list | `sudo apt update` | (implicit; metadata refreshed automatically) |
| Upgrade everything | `sudo apt upgrade` | `sudo dnf upgrade` |
| Install a package | `sudo apt install PKG` | `sudo dnf install PKG` |
| Install several | `sudo apt install PKG1 PKG2` | `sudo dnf install PKG1 PKG2` |
| Remove a package | `sudo apt remove PKG` | `sudo dnf remove PKG` |
| Remove + config | `sudo apt purge PKG` | `sudo dnf remove PKG` (same; configs not retained separately) |
| Search by name | `apt search WORD` | `dnf search WORD` |
| Show package info | `apt show PKG` | `dnf info PKG` |
| List installed | `apt list --installed` | `dnf list --installed` |
| Which package owns a file | `dpkg -S /path/to/file` | `dnf provides /path/to/file` (or `rpm -qf`) |
| Reinstall | `sudo apt install --reinstall PKG` | `sudo dnf reinstall PKG` |
| Auto-remove unused dependencies | `sudo apt autoremove` | `sudo dnf autoremove` |
| Clean cached downloads | `sudo apt clean` | `sudo dnf clean all` |
| List recent transactions | `cat /var/log/dpkg.log` | `dnf history` |
| Roll back a transaction | (not built-in; possible via `apt-get install pkg=version`) | `dnf history undo N` |

### Day-to-day workflow on Debian/Ubuntu

```bash
sudo apt update                             # refresh the package catalog (do this first!)
sudo apt upgrade                            # apply security and feature updates
sudo apt install htop                       # install one
sudo apt install nginx postgresql           # install several
sudo apt remove nginx                       # remove (keep config)
sudo apt purge nginx                        # remove + delete config files
sudo apt autoremove                         # clean up no-longer-needed dependencies
```

**`apt update` vs `apt upgrade`** confuses everyone at first:

- **`apt update`** — refreshes the **list of available packages** from the repositories. Doesn't install anything. Just learns what's out there.
- **`apt upgrade`** — actually **upgrades installed packages** based on what `update` learned.

Always run `apt update` before `apt upgrade` or `apt install`, or you'll be installing/upgrading based on a stale catalog.

### Day-to-day workflow on RHEL/Fedora

```bash
sudo dnf upgrade                            # upgrade everything (no separate `update` needed)
sudo dnf install htop                       # install one
sudo dnf install nginx postgresql-server    # install several
sudo dnf remove nginx                       # remove
sudo dnf autoremove                         # clean orphan dependencies
sudo dnf history                            # transaction history
sudo dnf history undo 42                    # roll back transaction 42 (LVM-snapshot-style undo)
```

`dnf` automatically refreshes metadata when it's stale (configurable in `/etc/dnf/dnf.conf`). No separate `update` step.

### The lower-level tools

When you have a `.deb` or `.rpm` file directly (downloaded outside repos), use the lower-level tools:

```bash
sudo dpkg -i package.deb                    # install a .deb file
sudo apt install -f                         # ...then this to fix any missing dependencies
dpkg -l                                     # list installed packages
dpkg -L PKG                                 # list files installed by PKG
dpkg -S /path/to/file                       # which package owns this file?

sudo rpm -i package.rpm                     # install an .rpm (rarely needed; dnf does this too)
rpm -qa                                     # list all installed packages
rpm -ql PKG                                 # list files in a package
rpm -qf /path/to/file                       # which package owns this file?
sudo dnf install ./package.rpm              # preferred: lets dnf resolve dependencies
```

For LFCS, **practice `apt` if you're on Ubuntu, `dnf` if you're on RHEL/Fedora**. The exam tests both styles depending on the platform.

In [ ]:
!command -v apt >/dev/null && apt --version || dnf --version 2>/dev/null

## Repositories — where packages come from

Packages don't materialise out of thin air. They come from **repositories** — servers with HTTP-accessible catalogs and package files. Your distro ships with a default set of repos enabled; you can add more.

### Debian/Ubuntu repositories

Configured in `/etc/apt/sources.list` and `/etc/apt/sources.list.d/*.list`. Each line:

```
deb http://archive.ubuntu.com/ubuntu jammy main restricted universe multiverse
```

Decoded:
- `deb` — binary packages (vs `deb-src` for source)
- `http://archive.ubuntu.com/ubuntu` — the URL
- `jammy` — the **codename** of the Ubuntu release (22.04 LTS in this case)
- `main restricted universe multiverse` — **components**:
  - **`main`** — officially supported FOSS
  - **`restricted`** — proprietary drivers
  - **`universe`** — community-supported FOSS
  - **`multiverse`** — proprietary software

To add a third-party repo (e.g. for Docker), you'd typically:

1. Add the repo's GPG key (so apt trusts its packages).
2. Add the repo source line to `/etc/apt/sources.list.d/`.
3. `sudo apt update` to fetch its catalog.
4. `sudo apt install PKG`.

### RHEL/Fedora repositories

Configured in `/etc/yum.repos.d/*.repo`. Each `.repo` file is INI-style:

```ini
[docker-ce-stable]
name=Docker CE Stable - $basearch
baseurl=https://download.docker.com/linux/centos/9/$basearch/stable
enabled=1
gpgcheck=1
gpgkey=https://download.docker.com/linux/centos/gpg
```

To enable / disable a repo:

```bash
sudo dnf config-manager --enable REPO_ID
sudo dnf config-manager --disable REPO_ID
sudo dnf repolist                           # see all repos
sudo dnf repolist --all                     # including disabled
```

Add a third-party repo with:

```bash
sudo dnf config-manager --add-repo URL_TO_REPO_FILE
```

### Why GPG signatures matter

Packages from real repositories are **signed** with the repo's GPG key. Your package manager verifies the signature before installing — protecting you from tampered packages on a compromised mirror. **Never ignore GPG errors.** Either install the key from a verified source or don't install the package.

### EPEL — the practical RHEL extra-packages repo

On RHEL/CentOS/Alma/Rocky, the default repos are conservative. **EPEL** ("Extra Packages for Enterprise Linux", maintained by Fedora) adds many community packages:

```bash
sudo dnf install epel-release               # adds the EPEL repo file
sudo dnf install htop                       # now available (wasn't in default RHEL repos)
```

EPEL is almost a "must install" on fresh RHEL-family systems. Not relevant on Ubuntu (the main + universe repos already cover most software).

For LFCS, know **where repo config lives** (`/etc/apt/sources.list*` vs `/etc/yum.repos.d/`), how to **add a repo**, and that **GPG keys gate trust**.